In [ ]:
import sys
sys.path.insert(0, '..')

from src.data import load_msd_metadata, load_spotify_tracks, match_datasets, dedup_tracks
from src.models import ContentModel

## Content-Based Demo

Rebuilds the content model from scratch using:
- Union of exact + fuzzy matches (~12.3K songs vs M1's 11.3K)
- `dedup_tracks()` to eliminate variant-track contamination before fitting

Expected: 'I Ran' query no longer returns 5 copies of 'I Ran' with similarity=1.0.

In [ ]:
spotify_df = load_spotify_tracks()
msd_meta   = load_msd_metadata()
print('Spotify tracks:', len(spotify_df))
print('MSD metadata songs:', len(msd_meta))

In [ ]:
# This takes several minutes -- fuzzy matching iterates over ~1M MSD songs.
matched = match_datasets(spotify_df, msd_meta, fuzzy_threshold=90)
print('Total matched rows (before dedup):', len(matched))
print('Unique MSD song_ids:', matched['song_id'].nunique())
print('Match type breakdown:')
print(matched['match_type'].value_counts())

In [ ]:
deduped = dedup_tracks(matched)
print('After dedup (one entry per title_clean+artist_clean):', len(deduped))

# Confirm 'I Ran' variants collapsed to one
i_ran_rows = deduped[
    (deduped['title_clean'] == 'i ran') &
    (deduped['artist_clean'] == 'a flock of seagulls')
]
print(f"\n'I Ran' by A Flock of Seagulls in deduped index: {len(i_ran_rows)} row (should be 1)")
display(i_ran_rows[['song_id', 'title', 'popularity']])

In [ ]:
content_model = ContentModel(n_neighbors=11)
content_model.fit(matched)  # fit() calls dedup_tracks internally
print('ContentModel fitted on', len(content_model._song_features), 'unique tracks.')

## Verify: 'I Ran' variant bug fixed

In [ ]:
# Find the canonical 'I Ran' entry (highest popularity after dedup)
i_ran_entry = content_model._song_features[
    (content_model._song_features['title_clean'] == 'i ran') &
    (content_model._song_features['artist_clean'] == 'a flock of seagulls')
]

if len(i_ran_entry) == 0:
    print("'I Ran' not in content index (not in matched set).")
else:
    seed_id = i_ran_entry.iloc[0]['song_id']
    print(f"Query: {i_ran_entry.iloc[0]['title']} -- {i_ran_entry.iloc[0]['artist_name']}")
    print()
    recs = content_model.recommend(seed_id, k=5)
    print('Top-5 recommendations (NO more variant-track contamination):')
    display(recs)
    
    # Confirm no other 'I Ran' variants in results
    i_ran_in_recs = recs['title'].str.lower().str.startswith('i ran').sum()
    print(f"\n'I Ran' variants in top-5: {i_ran_in_recs}  (was 5 in M1 -- bug fixed)")

## Save new model artifacts

In [ ]:
content_model.save('../models')
print('Saved: content_model.pkl, scaler.pkl, song_features.csv -> ../models/')